# Method zoo: восемь способов получить условный генератор за 1–4 вызова

Все восемь методов продолжают **одного и того же frozen conditional parent** своей задачи.
Это exploratory warm-start adaptations для toy data, а не буквальная репликация image papers.
CFM здесь тоже warm start; scratch baseline не подменяет основной cohort.

| Метод | Что видно в loss | Семплирование |
|---|---|---|
| CFM | velocity MSE на real-data linear bridge | Euler |
| OT-CFM | exact Hungarian coupling внутри одной condition | Euler |
| Endpoint | `f=x−t*v`, target — конец той же teacher trajectory | Euler; 2/4 шага помечены off-target |
| Progressive | два шага frozen stage teacher → один student step | matched stage 4/2/1 |
| Reflow | CFM на связанных noise→teacher endpoint pairs | Euler |
| Consistency | соседние teacher states → один EMA endpoint, RMS pseudo-Huber | denoise → re-noise |
| Shortcut | 75% FM + 25% two-half-step EMA bootstrap | interval-conditioned Euler |
| MeanFlow | 75% FM boundary + 25% average-velocity identity/JVP | interval-conditioned Euler |

Основной зарегистрированный cohort: **2000 updates на метод × 3 seeds × 2 задачи**.
Быстрый default ниже: 200 updates, один seed, rays; formulas и loops те же.
Progressive делит общий бюджет на пять стадий, а не получает пять полных бюджетов.
Обновления не равны FLOPs: отдельно сохраняются wall time, teacher/EMA/student forward
sample counts; JVP/backward входят во время, forward counts не выдаются за FLOPs.

Rays: target ID 0…15 передаётся в каждый вызов. Wall: condition — восемь floats
`[Axy,Bxy,wall_low_xy,wall_high_xy]`; один condition имеет оба обхода стены.
**Route ID не входит в модель.** Wall-state — 30 внутренних точек, A/B добавляются на каждом
сохранённом узле. Нулевой endpoint error задан конструкцией, не выучен.
Никакой geometry projection, collision filtering, clipping или CFG в этом сравнении нет.

Код обучения и формулы ниже на поверхности. Веса, private histories, optimizer states и wheel
в notebook не встроены. Final/test открывает отдельный evaluator только после фиксации cohort.

In [ ]:
import os
import subprocess
import sys
from importlib.metadata import version

from packaging.version import Version

GIT_BRANCH = "research/wasserstein-tube-replication"
MIN_VERSION = "0.5.0"
if os.environ.get("DIFFUSION_LAB_SKIP_INSTALL") != "1":
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--upgrade",
            "diffusion-lab[replication,interactive] @ git+https://github.com/Vovanm88/ddpm_vibe_research.git@"
            + GIT_BRANCH,
        ]
    )
if Version(version("diffusion-lab")) < Version(MIN_VERSION):
    raise RuntimeError("Install diffusion-lab >=0.5.0 from this branch; restart the kernel")

In [ ]:
import copy
import hashlib
import json
import math
import time
from contextlib import contextmanager
from pathlib import Path
from types import SimpleNamespace

import torch
from scipy.optimize import linear_sum_assignment
from tqdm.auto import tqdm

from diffusion_lab import PointDenoiser
from diffusion_lab.environments import make_environment
from diffusion_lab.environments.multimodal_navigation import make_multimodal_navigation
from diffusion_lab.interval_model import IntervalPointDenoiser, initialize_interval_from_parent
from recheck.fm import learning_rate
from recheck.runtime import (
    RNGStreams,
    atomic_json,
    choose_device,
    model_hash,
    save_checkpoint,
    stable_seed,
    tensor_hash,
)

In [ ]:
METHODS = (
    "cfm",
    "ot_cfm",
    "endpoint",
    "progressive",
    "reflow",
    "consistency",
    "shortcut",
    "meanflow",
)
TASKS = ("rays", "wall")
PARENTS = {
    "rays": {
        "path": "replication/demo/rays16_conditional_v1/teacher/ema_model",
        "sha256": "9bc0dbe39c4cb24c4d9b893bac084c89be66fdd7b29d6879fca786b4020d6ce1",
    },
    "wall": {
        "path": "replication/demo/wall_navigation_conditional_v1/teacher/ema_model",
        "sha256": "71d77760c37e6a053f324f2c810adb323439ed76ef51f09a57db43a5d2da9e79",
    },
}
DEFAULT_CONFIG = {
    "schema_version": 1,
    "protocol_id": "method_zoo_exploratory",
    "output_dir": "replication/artifacts/method_zoo_notebook15",
    "tasks": ["rays"],
    "methods": list(METHODS),
    "seeds": [161],
    "updates": 200,
    "batch_size": 128,
    "lr": 1e-4,
    "warmup": 20,
    "weight_decay": 1e-4,
    "ema_decay": 0.99,
    "grad_clip": 1.0,
    "device": "cpu",
    "threads": 4,
    "teacher_steps": 32,
    "pairbank_size": 8192,
    "pairbank_seed": 816001,
    "pairbank_batch_size": 256,
    "checkpoint_every": 100,
    "progress": True,
    "initialization": "parent",
    "progressive_steps": [16, 8, 4, 2, 1],
    "progressive_stage_warmup": 2,
    "consistency_step": 1 / 32,
    "consistency_huber": 0.03,
    "shortcut_fm_fraction": 0.75,
    "shortcut_widths": [1 / 16, 1 / 8, 1 / 4, 1 / 2, 1.0],
    "shortcut_smallest_half": 1 / 32,
    "meanflow_fm_fraction": 0.75,
    "meanflow_logit_mean": -0.4,
    "meanflow_logit_std": 1.0,
    "meanflow_adaptive_p": 1.0,
    "meanflow_adaptive_c": 0.001,
}


def resolved_config(config):
    if not isinstance(config, dict):
        raise ValueError("config must be an object")
    unknown = set(config) - set(DEFAULT_CONFIG)
    if unknown:
        raise ValueError("Unknown method-zoo config keys: " + ", ".join(sorted(unknown)))
    result = copy.deepcopy(DEFAULT_CONFIG)
    result.update(copy.deepcopy(config))
    if result["schema_version"] != 1:
        raise ValueError("schema_version must be 1")
    for name in (
        "updates",
        "batch_size",
        "threads",
        "teacher_steps",
        "pairbank_size",
        "pairbank_batch_size",
        "checkpoint_every",
    ):
        if isinstance(result[name], bool) or not isinstance(result[name], int) or result[name] < 1:
            raise ValueError(f"{name} must be a positive integer")
    for name in ("warmup", "progressive_stage_warmup"):
        if isinstance(result[name], bool) or not isinstance(result[name], int) or result[name] < 0:
            raise ValueError(f"{name} must be a nonnegative integer")
    for name in (
        "lr",
        "grad_clip",
        "consistency_step",
        "consistency_huber",
        "meanflow_logit_std",
        "meanflow_adaptive_c",
    ):
        if (
            isinstance(result[name], bool)
            or not isinstance(result[name], (int, float))
            or not math.isfinite(result[name])
            or result[name] <= 0
        ):
            raise ValueError(f"{name} must be finite and positive")
    for name in (
        "weight_decay",
        "ema_decay",
        "shortcut_fm_fraction",
        "meanflow_fm_fraction",
        "meanflow_adaptive_p",
        "meanflow_logit_mean",
    ):
        if (
            isinstance(result[name], bool)
            or not isinstance(result[name], (int, float))
            or not math.isfinite(result[name])
        ):
            raise ValueError(f"{name} must be finite")
    if (
        result["weight_decay"] < 0
        or not 0 <= result["ema_decay"] < 1
        or result["meanflow_adaptive_p"] < 0
    ):
        raise ValueError("Invalid weight_decay, ema_decay or adaptive exponent")
    if any(not 0 < result[key] < 1 for key in ("shortcut_fm_fraction", "meanflow_fm_fraction")):
        raise ValueError("FM fractions must be strictly between zero and one")
    for key, allowed in (("tasks", TASKS), ("methods", METHODS)):
        selected = result[key]
        if (
            not isinstance(selected, list)
            or not selected
            or len(set(selected)) != len(selected)
            or not set(selected) <= set(allowed)
        ):
            raise ValueError(f"{key} must be a nonempty unique selection from {allowed}")
    if (
        not isinstance(result["seeds"], list)
        or not result["seeds"]
        or len(set(result["seeds"])) != len(result["seeds"])
        or any(isinstance(s, bool) or not isinstance(s, int) or s < 0 for s in result["seeds"])
    ):
        raise ValueError("seeds must contain distinct nonnegative integers")
    if result["teacher_steps"] != 32 or result["progressive_steps"] != [16, 8, 4, 2, 1]:
        raise ValueError("This example uses teacher32 and progressive16/8/4/2/1")
    if result["consistency_step"] != 1 / 32:
        raise ValueError("Consistency positive grid and teacher step must be 1/32")
    if "progressive" in result["methods"] and (
        result["updates"] % 5 or result["progressive_stage_warmup"] >= result["updates"] // 5
    ):
        raise ValueError(
            "Progressive updates must divide into five stages with smaller stage warmup"
        )
    if result["warmup"] >= result["updates"]:
        raise ValueError("warmup must be smaller than updates")
    if result["batch_size"] < 4:
        raise ValueError("batch_size >=4 is needed for mixed FM/bootstrap batches")
    if (
        result["shortcut_widths"] != [1 / 16, 1 / 8, 1 / 4, 1 / 2, 1.0]
        or result["shortcut_smallest_half"] != 1 / 32
    ):
        raise ValueError(
            "Shortcut dyadic widths and smallest half-step must match the visible recipe"
        )
    if result["initialization"] not in ("parent", "scratch"):
        raise ValueError("initialization must be parent or scratch")
    if result["initialization"] == "scratch" and not set(result["methods"]) <= {
        "cfm",
        "ot_cfm",
        "shortcut",
        "meanflow",
    }:
        raise ValueError(
            "Scratch mode supports data-training methods cfm/ot_cfm/shortcut/meanflow only"
        )
    if result["device"] not in ("cpu", "cuda", "mps", "auto") or not isinstance(
        result["progress"], bool
    ):
        raise ValueError("Invalid device/progress")
    if (
        not isinstance(result["pairbank_seed"], int)
        or isinstance(result["pairbank_seed"], bool)
        or result["pairbank_seed"] < 0
    ):
        raise ValueError("pairbank_seed must be nonnegative integer")
    if (
        not isinstance(result["protocol_id"], str)
        or not result["protocol_id"]
        or not isinstance(result["output_dir"], str)
    ):
        raise ValueError("protocol_id/output_dir must be strings")
    return result


def json_digest(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()
    ).hexdigest()


def notebook_source_hash(repo_root):
    path = Path(repo_root) / "notebooks/15_conditional_method_zoo.ipynb"
    return hashlib.sha256(path.read_bytes()).hexdigest()


class ZooRNG(RNGStreams):
    names = ("data", "noise", "time", "condition", "pair", "duration", "preview")


def expand_time(t, x):
    return t.reshape(-1, *([1] * (x.ndim - 1)))


def call_velocity(model, x, t, conditions, width=None):
    if conditions is None:
        raise ValueError("Method zoo requires explicit target conditions; CFG is off")
    if isinstance(model, IntervalPointDenoiser):
        return model(x, t, conditions=conditions, step_size=0.0 if width is None else width).sample
    return model(x, t * 1000, conditions=conditions).sample


def model_task(model):
    task = getattr(model, "zoo_task", None)
    if task in TASKS:
        return task
    shape = tuple(model.sample_shape)
    if shape == (2,):
        return "rays"
    if shape == (30, 2):
        return "wall"
    raise ValueError("Set model.zoo_task for a supported task")


def full_state(x, conditions, task):
    if task == "rays":
        return x
    return torch.cat(
        (conditions[:, :2, None].transpose(1, 2), x, conditions[:, 2:4, None].transpose(1, 2)),
        dim=1,
    )

In [ ]:
def load_task(task, split="train"):
    if task not in TASKS or split not in ("train", "validation"):
        raise ValueError(
            "Training notebook supports rays/wall and train/validation only; final is gated separately"
        )
    if task == "rays":
        dataset = make_environment(
            "rotating_rays",
            n_samples=65536 if split == "train" else 8192,
            seed=47 if split == "train" else 10047,
            n_classes=16,
        )
        conditions = dataset.conditions
        groups = conditions
        states = dataset.samples
        condition_spec = {"kind": "categorical", "num_classes": 16}
    else:
        dataset = make_multimodal_navigation(
            n_scenes=128 if split == "train" else 32,
            samples_per_scene=128,
            path_length=32,
            seed=61 if split == "train" else 10061,
            split=split,
            jitter=0.025,
            clearance=0.06,
            margin=0.035,
        )
        conditions = dataset.conditions
        groups = dataset.scene_indices
        states = dataset.samples[:, 1:-1]
        condition_spec = {"kind": "continuous", "dimension": 8, "route_id_input": False}
    pools = [torch.where(groups == index)[0] for index in range(int(groups.max()) + 1)]
    return {
        "task": task,
        "split": split,
        "samples": states.contiguous(),
        "raw_samples": dataset.samples,
        "conditions": conditions,
        "groups": groups,
        "pools": pools,
        "sample_shape": tuple(states.shape[1:]),
        "condition_spec": condition_spec,
        "train_data_sha256": tensor_hash(dataset.samples),
        "train_state_sha256": tensor_hash(states),
        "train_conditions_sha256": tensor_hash(conditions),
        "endpoint_clamped_contract": task == "wall",
    }


def load_parent(task, data, repo_root, device):
    path = Path(repo_root) / PARENTS[task]["path"]
    manifest = json.loads((path.parent / "manifest.json").read_text())
    native = json.loads((path / "config.json").read_text())
    if manifest.get("status") != "completed" or manifest.get("updates_completed") != 20000:
        raise ValueError("Expected the completed 20000-update conditional parent")
    if (
        manifest.get("teacher_sha256") != PARENTS[task]["sha256"]
        or native.get("_class_name") != "PointDenoiser"
    ):
        raise ValueError("Wrong native parent or parent SHA")
    if manifest.get("data_sha256") != data["train_data_sha256"]:
        raise ValueError("Parent training-data SHA differs")
    expected_conditioner = "embedding" if task == "rays" else "mlp"
    if (
        native.get("conditioner") != expected_conditioner
        or tuple(native["sample_shape"]) != data["sample_shape"]
    ):
        raise ValueError("Wrong conditional model profile")
    if task == "rays" and native.get("num_classes") != 16:
        raise ValueError("Expected sixteen target labels")
    if task == "wall" and (
        native.get("condition_dim") != 8 or not manifest.get("endpoint_clamped_contract")
    ):
        raise ValueError(
            "Wall parent must predict interior30x2 with continuous8 and explicit fixed endpoints"
        )
    teacher = (
        PointDenoiser.from_pretrained(path, local_files_only=True, low_cpu_mem_usage=False)
        .to(device)
        .eval()
        .requires_grad_(False)
    )
    teacher.zoo_task = task
    teacher.zoo_method = "teacher"
    if model_hash(teacher) != PARENTS[task]["sha256"]:
        raise ValueError("Actual parent tensor SHA differs")
    return teacher, manifest


def load_zoo_model(export_dir, *, device="cpu"):
    export_dir = Path(export_dir)
    native = json.loads((export_dir / "config.json").read_text())
    manifest = json.loads((export_dir.parent / "manifest.json").read_text())
    if manifest.get("status") != "completed":
        raise ValueError("Native export is not completed")
    classes = {"PointDenoiser": PointDenoiser, "IntervalPointDenoiser": IntervalPointDenoiser}
    if native.get("_class_name") not in classes:
        raise ValueError("Unsupported native model class")
    model = (
        classes[native["_class_name"]]
        .from_pretrained(export_dir, local_files_only=True, low_cpu_mem_usage=False)
        .to(device)
        .eval()
        .requires_grad_(False)
    )
    expected = manifest.get("ema_sha256", manifest.get("teacher_sha256"))
    if expected is None or model_hash(model) != expected:
        raise ValueError("Native EMA tensor SHA differs")
    model.zoo_task = manifest["task"] if "task" in manifest else model_task(model)
    model.zoo_method = manifest.get("method", "teacher")
    return model

## Sampling: считать реальные вызовы и сохранять реальные события

Для обычных velocity и interval моделей — Euler с `t=1→0`, ровно K calls.
Для consistency каждый call предсказывает endpoint `f=x−t*v`; затем (кроме последнего)
следующее состояние `(1−s)*f+s*extra_noise`. Его история **не монотонна по t**:
input at t → predicted endpoint at 0 → next noisy input at s. Возвращаются отдельно
`actual_inputs`, `postcall_outputs` и event times. Дополнительный шум передаётся явно;
один bank можно повторить для всех conditions и методов/seed. Стеновые A/B добавляются
на каждом событии; исходные interior states также возвращаются.

In [ ]:
@torch.no_grad()
def sample_zoo(
    model,
    initial_noise,
    conditions,
    *,
    nfe,
    batch_size=128,
    record_history=True,
    extra_noise=None,
    generator=None,
):
    if isinstance(nfe, bool) or not isinstance(nfe, int) or nfe < 1:
        raise ValueError("nfe must be a positive integer")
    if (
        batch_size < 1
        or len(initial_noise) < 1
        or tuple(initial_noise.shape[1:]) != tuple(model.sample_shape)
    ):
        raise ValueError("Wrong noise shape or batch_size")
    if conditions is None or len(conditions) != len(initial_noise):
        raise ValueError("Explicit batched conditions are required")
    task = model_task(model)
    method = getattr(model, "zoo_method", "teacher")
    if task == "rays" and (
        conditions.shape != (len(initial_noise),)
        or conditions.dtype not in (torch.int32, torch.int64)
    ):
        raise ValueError("Rays require integer target IDs[B]")
    if task == "wall" and (
        conditions.shape != (len(initial_noise), 8) or not conditions.is_floating_point()
    ):
        raise ValueError("Wall requires continuous conditions[B,8]")
    if not torch.isfinite(initial_noise).all() or not torch.isfinite(conditions).all():
        raise ValueError("Nonfinite sampling input")
    if method == "consistency" and nfe > 1:
        expected = (nfe - 1, *initial_noise.shape)
        if extra_noise is None:
            if generator is None:
                raise ValueError("Consistency K>1 requires explicit extra_noise or a generator")
            extra_noise = torch.randn(
                expected, generator=generator, device="cpu", dtype=initial_noise.dtype
            )
        if tuple(extra_noise.shape) != expected or not torch.isfinite(extra_noise).all():
            raise ValueError("extra_noise must be finite[K-1,B,*state]")
    was_training = model.training
    model.eval()
    outputs, histories, interior_histories, actual_inputs, postcall_outputs = [], [], [], [], []
    event_times, event_kinds = None, None
    forward_calls = 0
    try:
        for start in range(0, len(initial_noise), batch_size):
            stop = min(start + batch_size, len(initial_noise))
            state = initial_noise[start:stop].to(device=model.device, dtype=model.dtype).clone()
            cond = conditions[start:stop].to(model.device)
            history = []
            interior = []
            inputs = []
            post = []
            times = []
            kinds = []
            if method != "consistency" and record_history:
                history.append(full_state(state, cond, task).cpu().clone())
                interior.append(state.cpu().clone())
                times.append(1.0)
                kinds.append("initial_noise")
            for step in range(nfe):
                t = 1 - step / nfe
                if method == "consistency":
                    if record_history:
                        frame = full_state(state, cond, task).cpu().clone()
                        inputs.append(frame)
                        history.append(frame)
                        interior.append(state.cpu().clone())
                        times.append(t)
                        kinds.append("model_input")
                    prediction = state - t * call_velocity(model, state, t, cond)
                    forward_calls += 1
                    if record_history:
                        frame = full_state(prediction, cond, task).cpu().clone()
                        post.append(frame)
                        history.append(frame)
                        interior.append(prediction.cpu().clone())
                        times.append(0.0)
                        kinds.append("denoised_endpoint")
                    if step + 1 < nfe:
                        s = 1 - (step + 1) / nfe
                        fresh = extra_noise[step, start:stop].to(
                            device=model.device, dtype=model.dtype
                        )
                        state = (1 - s) * prediction + s * fresh
                    else:
                        state = prediction
                else:
                    state = state - call_velocity(model, state, t, cond, width=1 / nfe) / nfe
                    forward_calls += 1
                    if record_history:
                        history.append(full_state(state, cond, task).cpu().clone())
                        interior.append(state.cpu().clone())
                        times.append(1 - (step + 1) / nfe)
                        kinds.append("euler_state")
                if not torch.isfinite(state).all():
                    raise FloatingPointError("Nonfinite raw generated state")
            outputs.append(full_state(state, cond, task).cpu())
            if record_history:
                histories.append(torch.stack(history))
                interior_histories.append(torch.stack(interior))
                if inputs:
                    actual_inputs.append(torch.stack(inputs))
                    postcall_outputs.append(torch.stack(post))
                event_times, event_kinds = times, kinds
    finally:
        model.train(was_training)
    return {
        "samples": torch.cat(outputs),
        "history": torch.cat(histories, dim=1) if histories else None,
        "interior_history": torch.cat(interior_histories, dim=1) if interior_histories else None,
        "actual_inputs": torch.cat(actual_inputs, dim=1) if actual_inputs else None,
        "postcall_outputs": torch.cat(postcall_outputs, dim=1) if postcall_outputs else None,
        "times": torch.tensor(event_times) if event_times else None,
        "event_history": torch.cat(histories, dim=1) if histories else None,
        "event_times": torch.tensor(event_times) if event_times else None,
        "event_kinds": event_kinds,
        "input_times": torch.arange(nfe, 0, -1) / nfe if method == "consistency" else None,
        "output_times": torch.zeros(nfe) if method == "consistency" else None,
        "nfe": nfe,
        "model_calls": forward_calls,
        "model_sample_evals": nfe * len(initial_noise),
        "sampler": "denoise_renoise" if method == "consistency" else "euler",
        "initial_noise_sha256": tensor_hash(initial_noise),
        "conditions_sha256": tensor_hash(conditions),
        "extra_noise_sha256": tensor_hash(extra_noise)
        if method == "consistency" and nfe > 1
        else None,
        "endpoint_clamped_contract": task == "wall",
    }

In [ ]:
@contextmanager
def count_forwards(model, counters, prefix):
    def record(module, args):
        counters[prefix + "_forward_calls"] = counters.get(prefix + "_forward_calls", 0) + 1
        counters[prefix + "_sample_evals"] = counters.get(prefix + "_sample_evals", 0) + len(
            args[0]
        )

    hook = model.register_forward_pre_hook(record)
    try:
        yield
    finally:
        hook.remove()


def bank_contract(data, teacher, config):
    if data.get("split") != "train":
        raise ValueError("Teacher pairbank accepts train data only")
    return {
        "task": data["task"],
        "split": "train",
        "teacher_sha256": model_hash(teacher),
        "train_data_sha256": data["train_data_sha256"],
        "train_conditions_sha256": data["train_conditions_sha256"],
        "size": config["pairbank_size"],
        "seed": config["pairbank_seed"],
        "teacher_steps": config["teacher_steps"],
    }


def prepare_pairbank(data, teacher, config, bank_dir):
    # The lock guards simultaneous CLI workers. A valid cache is never rewritten.
    import fcntl

    bank_dir = Path(bank_dir)
    bank_dir.mkdir(parents=True, exist_ok=True)
    expected = bank_contract(data, teacher, config)
    with (bank_dir / ".prepare.lock").open("a") as lock:
        fcntl.flock(lock, fcntl.LOCK_EX)
        payload_path = bank_dir / "pairs.pt"
        metadata_path = bank_dir / "manifest.json"
        if payload_path.exists() or metadata_path.exists():
            if not payload_path.exists() or not metadata_path.exists():
                raise ValueError("Incomplete pairbank; preserve it and use a new output directory")
            metadata = json.loads(metadata_path.read_text())
            if metadata["contract"] != expected:
                raise ValueError("Frozen pairbank contract differs")
            bank = torch.load(payload_path, map_location="cpu", weights_only=True)
            for key in ("history", "noise", "conditions", "condition_ids"):
                if tensor_hash(bank[key]) != metadata["hashes"][key]:
                    raise ValueError("Frozen pairbank tensor hash differs: " + key)
            return bank, metadata
        count = len(data["pools"])
        if config["pairbank_size"] % count:
            raise ValueError("pairbank_size must be divisible by the number of train conditions")
        per_condition = config["pairbank_size"] // count
        condition_ids = torch.arange(count).repeat_interleave(per_condition)
        conditions = torch.cat(
            [
                data["conditions"][pool[:1]].expand(per_condition, *data["conditions"].shape[1:])
                for pool in data["pools"]
            ]
        )
        generator = torch.Generator().manual_seed(
            stable_seed(config["pairbank_seed"], data["task"])
        )
        noise = torch.randn(config["pairbank_size"], *data["sample_shape"], generator=generator)
        costs = {}
        started = time.perf_counter()
        with count_forwards(teacher, costs, "teacher"):
            result = sample_zoo(
                teacher,
                noise,
                conditions,
                nfe=config["teacher_steps"],
                batch_size=config["pairbank_batch_size"],
                record_history=True,
            )
        bank = {
            "history": result["interior_history"],
            "noise": noise,
            "conditions": conditions,
            "condition_ids": condition_ids,
        }
        metadata = {
            "status": "completed",
            "contract": expected,
            "hashes": {k: tensor_hash(v) for k, v in bank.items()},
            "generation_seconds": time.perf_counter() - started,
            "generation_cost": costs,
            "quality_filtering": False,
            "reuse": "same fixed bank across all methods and seeds",
        }
        metadata["sha256"] = json_digest(metadata["contract"] | metadata["hashes"])
        save_checkpoint(payload_path, bank)
        atomic_json(metadata_path, metadata)
        return bank, metadata

## Общая Torch-механика: optimizer, EMA, checkpoints

Следующая ячейка содержит обычные `loss.backward()`, gradient clipping, `optimizer.step()`
и EMA. Ниже каждый метод сам строит batch, target и loss в отдельном видимом цикле.
Два независимых counterfamily: live student и frozen targets (parent/EMA/stage teacher).
Python forward hooks считают реальные вызовы и размер входного batch; это **не FLOPs**.
Resume восстанавливает live/EMA/AdamW, все RNG streams и текущую progressive stage.

In [ ]:
def clone_model_without_hooks(model):
    # Instrumentation callbacks must not be inherited by a new student/target.
    result = copy.deepcopy(model)
    for module in result.modules():
        module._forward_pre_hooks.clear()
        module._forward_hooks.clear()
    return result


def optimizer_for(model, config):
    return torch.optim.AdamW(
        model.parameters(), lr=config["lr"], betas=(0.9, 0.999), weight_decay=config["weight_decay"]
    )


def training_contract(method, data, teacher, seed, config, source_sha256, bank_metadata):
    semantic = {
        k: v
        for k, v in config.items()
        if k not in ("output_dir", "device", "threads", "progress", "tasks", "methods", "seeds")
    }
    return {
        "schema_version": 1,
        "task": data["task"],
        "method": method,
        "seed": seed,
        "teacher_sha256": model_hash(teacher),
        "train_data_sha256": data["train_data_sha256"],
        "train_conditions_sha256": data["train_conditions_sha256"],
        "train_state_sha256": data["train_state_sha256"],
        "source_sha256": source_sha256,
        "config": semantic,
        "pairbank_sha256": bank_metadata["sha256"],
    }


def refresh_hooks(ctx):
    for hook in ctx.hooks:
        hook.remove()
    ctx.hooks = []
    for model, prefix in ((ctx.model, "student"), (ctx.ema, "ema"), (ctx.teacher, "teacher")):

        def record(module, args, prefix=prefix):
            ctx.costs[prefix + "_forward_calls"] = ctx.costs.get(prefix + "_forward_calls", 0) + 1
            ctx.costs[prefix + "_sample_evals"] = ctx.costs.get(prefix + "_sample_evals", 0) + len(
                args[0]
            )

        ctx.hooks.append(model.register_forward_pre_hook(record))
    if ctx.stage_teacher is not None:

        def record_stage(module, args):
            ctx.costs["stage_teacher_forward_calls"] = (
                ctx.costs.get("stage_teacher_forward_calls", 0) + 1
            )
            ctx.costs["stage_teacher_sample_evals"] = ctx.costs.get(
                "stage_teacher_sample_evals", 0
            ) + len(args[0])

        ctx.hooks.append(ctx.stage_teacher.register_forward_pre_hook(record_stage))


def start_context(
    method,
    data,
    teacher,
    seed,
    config,
    directory,
    source_sha256,
    bank,
    bank_metadata,
    resume=False,
    stop_after=None,
):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    interval = method in ("shortcut", "meanflow")
    initial_parent = teacher
    if config["initialization"] == "scratch":
        keys = (
            "sample_shape",
            "hidden_dim",
            "num_layers",
            "num_heads",
            "backbone",
            "conditioner",
            "num_classes",
            "condition_dim",
        )
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(seed)
            initial_parent = PointDenoiser(**{key: teacher.config[key] for key in keys}).to(
                teacher.device
            )
    model = (
        initialize_interval_from_parent(initial_parent, seed=seed, allow_null=False)
        if interval
        else clone_model_without_hooks(initial_parent)
    )
    model.train().requires_grad_(True)
    model.zoo_task = data["task"]
    model.zoo_method = method
    ema = clone_model_without_hooks(model).eval().requires_grad_(False)
    contract = training_contract(method, data, teacher, seed, config, source_sha256, bank_metadata)
    ctx = SimpleNamespace(
        method=method,
        data=data,
        teacher=teacher,
        model=model,
        ema=ema,
        optimizer=optimizer_for(model, config),
        rng=ZooRNG(seed),
        config=config,
        directory=directory,
        contract=contract,
        completed=0,
        history=[],
        raw_trace=[],
        pair_trace=[],
        phase=-1,
        stage_teacher=None,
        costs={},
        hooks=[],
        bank=bank,
        bank_metadata=bank_metadata,
        seed=seed,
        init_sha256=model_hash(model),
        elapsed=0.0,
        started=time.perf_counter(),
        limit=min(config["updates"], stop_after) if stop_after is not None else config["updates"],
    )
    if ctx.limit < 1:
        raise ValueError("stop_after must allow at least one update")
    path = directory / "training.pt"
    if path.exists():
        if not resume:
            raise FileExistsError("Use a new output directory or resume=True")
        saved = torch.load(path, map_location="cpu", weights_only=True)
        if saved["contract"] != contract:
            raise ValueError("Method-zoo resume contract differs")
        ctx.model.load_state_dict(saved["model"])
        ctx.ema.load_state_dict(saved["ema"])
        ctx.optimizer.load_state_dict(saved["optimizer"])
        for state in ctx.optimizer.state.values():
            for key, value in state.items():
                if isinstance(value, torch.Tensor):
                    state[key] = value.to(model.device)
        ctx.rng.load_state_dict(saved["rng"])
        for name in (
            "completed",
            "history",
            "raw_trace",
            "pair_trace",
            "phase",
            "costs",
            "elapsed",
            "init_sha256",
        ):
            setattr(ctx, name, saved[name])
        if saved["stage_teacher"] is not None:
            ctx.stage_teacher = clone_model_without_hooks(teacher).eval().requires_grad_(False)
            ctx.stage_teacher.load_state_dict(saved["stage_teacher"])
    refresh_hooks(ctx)
    return ctx


def checkpoint_context(ctx, status):
    save_checkpoint(
        ctx.directory / "training.pt",
        {
            "contract": ctx.contract,
            "model": ctx.model.state_dict(),
            "ema": ctx.ema.state_dict(),
            "optimizer": ctx.optimizer.state_dict(),
            "rng": ctx.rng.state_dict(),
            "completed": ctx.completed,
            "history": ctx.history,
            "raw_trace": ctx.raw_trace,
            "pair_trace": ctx.pair_trace,
            "phase": ctx.phase,
            "stage_teacher": ctx.stage_teacher.state_dict()
            if ctx.stage_teacher is not None
            else None,
            "costs": ctx.costs,
            "elapsed": ctx.elapsed + time.perf_counter() - ctx.started,
            "init_sha256": ctx.init_sha256,
            "status": status,
        },
    )


def data_batch(ctx):
    rng = ctx.rng
    data = ctx.data
    batch = ctx.config["batch_size"]
    condition_id = int(torch.randint(len(data["pools"]), (), generator=rng["condition"]))
    pool = data["pools"][condition_id]
    indices = pool[torch.randint(len(pool), (batch,), generator=rng["data"])]
    clean = data["samples"][indices].to(ctx.model.device)
    conditions = data["conditions"][indices].to(ctx.model.device)
    noise = torch.randn(clean.shape, generator=rng["noise"]).to(ctx.model.device)
    t = torch.rand(batch, generator=rng["time"]).to(ctx.model.device)
    assert torch.equal(conditions, conditions[:1].expand_as(conditions))
    ctx.raw_trace.append(
        json_digest(
            {
                "condition_id": condition_id,
                "indices": tensor_hash(indices),
                "noise": tensor_hash(noise),
                "uniform_t": tensor_hash(t),
            }
        )
    )
    return clean, noise, t, conditions


def pair_batch(ctx):
    count = len(ctx.data["pools"])
    batch = ctx.config["batch_size"]
    condition_id = int(torch.randint(count, (), generator=ctx.rng["condition"]))
    pool = torch.where(ctx.bank["condition_ids"] == condition_id)[0]
    indices = pool[torch.randint(len(pool), (batch,), generator=ctx.rng["pair"])]
    ctx.pair_trace.append(
        json_digest({"condition_id": condition_id, "indices": tensor_hash(indices)})
    )
    return indices, ctx.bank["conditions"][indices].to(ctx.model.device)


def take_step(ctx, loss, details=None, *, local_update=None, local_total=None, local_warmup=None):
    if not torch.isfinite(loss):
        raise FloatingPointError("Nonfinite method-zoo loss")
    update = ctx.completed if local_update is None else local_update
    total = ctx.config["updates"] if local_total is None else local_total
    warmup = ctx.config["warmup"] if local_warmup is None else local_warmup
    lr = learning_rate(update, total, warmup, ctx.config["lr"])
    for group in ctx.optimizer.param_groups:
        group["lr"] = lr
    ctx.optimizer.zero_grad(set_to_none=True)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(
        ctx.model.parameters(), ctx.config["grad_clip"], error_if_nonfinite=True
    )
    ctx.optimizer.step()
    with torch.no_grad():
        for average, parameter in zip(ctx.ema.parameters(), ctx.model.parameters()):
            average.lerp_(parameter.detach(), 1 - ctx.config["ema_decay"])
        for average, buffer in zip(ctx.ema.buffers(), ctx.model.buffers()):
            average.copy_(buffer)
    ctx.completed += 1
    row = {
        "update": ctx.completed,
        "loss": float(loss.detach()),
        "grad_norm": float(grad_norm),
        "lr": lr,
    }
    row.update(details or {})
    ctx.history.append(row)
    if ctx.completed == 1 or ctx.completed % ctx.config["checkpoint_every"] == 0:
        checkpoint_context(ctx, "running")


def update_range(ctx):
    return tqdm(
        range(ctx.completed, ctx.limit),
        desc=ctx.method,
        disable=not ctx.config["progress"],
        miniters=1,
    )


def export_model(ctx, directory, *, trained_nfe=None, cumulative_updates=None):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    ctx.ema.save_pretrained(directory / "ema_model", safe_serialization=True)
    metadata = {
        "schema_version": 1,
        "status": "completed",
        "task": ctx.data["task"],
        "method": ctx.method,
        "seed": ctx.seed,
        "ema_sha256": model_hash(ctx.ema),
        "teacher_sha256": ctx.contract["teacher_sha256"],
        "source_sha256": ctx.contract["source_sha256"],
        "trained_nfe": trained_nfe,
        "cumulative_updates_at_export": ctx.completed
        if cumulative_updates is None
        else cumulative_updates,
    }
    atomic_json(directory / "manifest.json", metadata)
    return metadata

## 1–2. Continued CFM и condition-local OT-CFM

Одинаковые parent weights, raw data/noise/time RNG для двух методов. Hungarian меняет
**перестановку того же noise batch** по squared Euclidean cost полного state.
Весь batch имеет одну condition; для wall top/bottom labels не читаются. Это minibatch
OT coupling, не global OT между всеми scenes и не Sinkhorn regularization loss.

In [ ]:
def train_cfm(ctx, *, optimal_transport=False):
    for _ in update_range(ctx):
        clean, noise, t, conditions = data_batch(ctx)
        details = {}
        if optimal_transport:
            dimension = clean[0].numel()
            cost = (
                torch.cdist(
                    clean.flatten(1).detach().cpu().double(),
                    noise.flatten(1).detach().cpu().double(),
                ).square()
                / dimension
            )
            rows, columns = linear_sum_assignment(cost.numpy())
            assert (rows == torch.arange(len(clean)).numpy()).all()
            details["assignment_cost"] = float(cost[rows, columns].mean())
            details["uncoupled_cost"] = float(cost.diagonal().mean())
            noise = noise[torch.as_tensor(columns, device=noise.device)]
        anchor = (1 - expand_time(t, clean)) * clean + expand_time(t, clean) * noise
        target = noise - clean
        predicted = call_velocity(ctx.model, anchor, t, conditions)
        loss = (predicted - target).square().mean()
        take_step(ctx, loss, details)

## 3. Endpoint regression: один конец для всех положительных teacher nodes

Из общей fixed train-only bank выбирается trajectory и node `t=1,…,1/32`.
`f_theta(x_t,t)=x_t−t*v_theta(x_t,t)` сравнивается с endpoint **той же trajectory**.
Это endpoint-coordinate MSE; умножение на `1/t²` не спрятано в helper.

In [ ]:
def train_endpoint(ctx):
    for _ in update_range(ctx):
        indices, conditions = pair_batch(ctx)
        nodes = torch.randint(
            ctx.config["teacher_steps"], (len(indices),), generator=ctx.rng["time"]
        )
        anchor = ctx.bank["history"][nodes, indices].to(ctx.model.device)
        endpoint = ctx.bank["history"][-1, indices].to(ctx.model.device)
        t = (1 - nodes.float() / ctx.config["teacher_steps"]).to(ctx.model.device)
        prediction = anchor - expand_time(t, anchor) * call_velocity(
            ctx.model, anchor, t, conditions
        )
        loss = (prediction - endpoint).square().mean()
        take_step(ctx, loss)

## 4. Progressive: 32→16→8→4→2→1

Каждая стадия имеет `total_updates/5`. В её начале live student **и EMA** копируют frozen
предыдущую stage EMA; AdamW создаётся заново, cosine и warmup локальны стадии.
Для main 2000: 5×400 updates и warmup 20 на стадию. Quick 200: 5×40 и warmup 2.
Teacher target — **два текущих stage-teacher Euler steps**, не 32 исходных шага каждый раз.
NFE4/2/1 exports имеют cumulative 1200/1600/2000 updates в main; final holdout всё равно
ждёт завершения полного cohort. Stage checkpoints выбираются расписанием, не качеством.

In [ ]:
def train_progressive(ctx):
    stages = ctx.config["progressive_steps"]
    stage_updates = ctx.config["updates"] // len(stages)
    # A process may stop after the boundary checkpoint but before its native export.
    # The saved EMA is already the exact completed stage result; recover that export.
    if ctx.phase >= 0 and ctx.completed == (ctx.phase + 1) * stage_updates:
        completed_nfe = stages[ctx.phase]
        export_model(
            ctx,
            ctx.directory / f"stage{completed_nfe}",
            trained_nfe=completed_nfe,
            cumulative_updates=ctx.completed,
        )
    for phase, student_nfe in enumerate(stages):
        stage_start = phase * stage_updates
        stage_end = (phase + 1) * stage_updates
        if ctx.completed >= stage_end:
            continue
        if ctx.completed >= ctx.limit:
            break
        if ctx.phase != phase:
            if phase == 0:
                target = clone_model_without_hooks(ctx.teacher)
            else:
                target = load_zoo_model(
                    ctx.directory / f"stage{stages[phase - 1]}" / "ema_model",
                    device=ctx.model.device,
                )
            ctx.stage_teacher = target.eval().requires_grad_(False)
            ctx.model = clone_model_without_hooks(target).train().requires_grad_(True)
            ctx.ema = clone_model_without_hooks(target).eval().requires_grad_(False)
            ctx.optimizer = optimizer_for(ctx.model, ctx.config)
            ctx.phase = phase
            refresh_hooks(ctx)
        width = 1 / student_nfe
        half = width / 2
        iterator = tqdm(
            range(ctx.completed, min(stage_end, ctx.limit)),
            desc=f"progressive N{student_nfe}",
            disable=not ctx.config["progress"],
            miniters=1,
        )
        for _ in iterator:
            clean, noise, _, conditions = data_batch(ctx)
            nodes = torch.randint(student_nfe, (len(clean),), generator=ctx.rng["time"])
            t = (1 - nodes.float() / student_nfe).to(ctx.model.device)
            anchor = (1 - expand_time(t, clean)) * clean + expand_time(t, clean) * noise
            with torch.no_grad():
                midpoint = anchor - half * call_velocity(ctx.stage_teacher, anchor, t, conditions)
                target = midpoint - half * call_velocity(
                    ctx.stage_teacher, midpoint, t - half, conditions
                )
            prediction = anchor - width * call_velocity(ctx.model, anchor, t, conditions)
            loss = (prediction - target).square().mean()
            take_step(
                ctx,
                loss,
                {"stage_nfe": student_nfe},
                local_update=ctx.completed - stage_start,
                local_total=stage_updates,
                local_warmup=ctx.config["progressive_stage_warmup"],
            )
        if ctx.completed == stage_end:
            export_model(
                ctx,
                ctx.directory / f"stage{student_nfe}",
                trained_nfe=student_nfe,
                cumulative_updates=stage_end,
            )
            checkpoint_context(ctx, "running")

## 5. Один reflow round на связанных парах

Банк и trajectory indices те же, что у endpoint regression. Меняется objective:
прямой bridge между исходным Gaussian и его parent endpoint, target `noise−endpoint`.
Gaussian и endpoint не перемешиваются независимо. Никакой quality filtering банка нет.

In [ ]:
def train_reflow(ctx):
    for _ in update_range(ctx):
        indices, conditions = pair_batch(ctx)
        noise = ctx.bank["noise"][indices].to(ctx.model.device)
        endpoint = ctx.bank["history"][-1, indices].to(ctx.model.device)
        t = torch.rand(len(indices), generator=ctx.rng["time"]).to(ctx.model.device)
        anchor = (1 - expand_time(t, noise)) * endpoint + expand_time(t, noise) * noise
        target = noise - endpoint
        prediction = call_velocity(ctx.model, anchor, t, conditions)
        loss = (prediction - target).square().mean()
        take_step(ctx, loss)

## 6. Flow consistency distillation adaptation

`h=1/32`, positive grid `t=j/32`. Teacher делает один Euler step до `s=t−h`.
Target — detached EMA endpoint `f_EMA(x_s,s)`, prediction — live endpoint `f_theta(x_t,t)`.
Boundary `f(x,0)=x` задана параметризацией. Loss —
`sqrt(mean_D((prediction−target)²)+0.03²)−0.03`, затем mean по batch.
Sampling этого метода выше реализует denoise/re-noise, а не выдаёт обычный Euler за CM.

In [ ]:
def train_consistency(ctx):
    h = ctx.config["consistency_step"]
    for _ in update_range(ctx):
        clean, noise, _, conditions = data_batch(ctx)
        nodes = torch.randint(1, 33, (len(clean),), generator=ctx.rng["time"])
        t = (nodes.float() / 32).to(ctx.model.device)
        s = t - h
        anchor = (1 - expand_time(t, clean)) * clean + expand_time(t, clean) * noise
        with torch.no_grad():
            next_state = anchor - h * call_velocity(ctx.teacher, anchor, t, conditions)
            target = next_state - expand_time(s, next_state) * call_velocity(
                ctx.ema, next_state, s, conditions
            )
        prediction = anchor - expand_time(t, anchor) * call_velocity(
            ctx.model, anchor, t, conditions
        )
        mse_per_sample = (prediction - target).flatten(1).square().mean(dim=1)
        c = ctx.config["consistency_huber"]
        loss = (torch.sqrt(mse_per_sample + c * c) - c).mean()
        take_step(ctx, loss)

## 7. Shortcut: FM boundary и настоящий midpoint bootstrap

75% batch: `d=0`, обычный FM target. Остальные 25%: coarse width `D` из
`{1/16,1/8,1/4,1/2,1}`, half `h=D/2`. Frozen EMA даёт `u1`, затем реальный midpoint
`x−h*u1`, затем `u2`; target `(u1+u2)/2`. Когда physical half равен `1/32`, duration
query равна **0**, потому что это базовый FM boundary. Physical midpoint всё равно
сдвигается на h. Condition сохраняется на обоих вызовах. Native model получает t,d
в нормализованных единицах; дополнительная duration branch изначально нейтральна.

In [ ]:
def train_shortcut(ctx):
    for _ in update_range(ctx):
        clean, noise, t, conditions = data_batch(ctx)
        batch = len(clean)
        fm_count = int(batch * ctx.config["shortcut_fm_fraction"])
        widths = torch.tensor(
            ctx.config["shortcut_widths"], device=ctx.model.device, dtype=clean.dtype
        )
        chosen = torch.randint(len(widths), (batch - fm_count,), generator=ctx.rng["duration"]).to(
            clean.device
        )
        coarse = widths[chosen]
        # Valid dyadic start nodes t >= D; drawing on CPU retains explicit RNG control.
        uniforms = torch.rand(batch - fm_count, generator=ctx.rng["duration"]).to(clean.device)
        t[fm_count:] = (torch.floor(uniforms / coarse) + 1) * coarse
        duration = torch.zeros_like(t)
        duration[fm_count:] = coarse
        anchor = (1 - expand_time(t, clean)) * clean + expand_time(t, clean) * noise
        target = (noise - clean).detach().clone()
        with torch.no_grad():
            x = anchor[fm_count:]
            cond = conditions[fm_count:]
            tt = t[fm_count:]
            half = coarse / 2
            query = torch.where(
                half <= ctx.config["shortcut_smallest_half"], torch.zeros_like(half), half
            )
            u1 = call_velocity(ctx.ema, x, tt, cond, width=query)
            midpoint = x - expand_time(half, x) * u1
            u2 = call_velocity(ctx.ema, midpoint, tt - half, cond, width=query)
            target[fm_count:] = (u1 + u2) / 2
        prediction = call_velocity(ctx.model, anchor, t, conditions, width=duration)
        loss = (prediction - target).square().mean()
        take_step(ctx, loss, {"fm_examples": fm_count, "bootstrap_examples": batch - fm_count})

## 8. MeanFlow: JVP с правильными native coordinates

Для бумажной записи `u(x,r,t)` держим r фиксированным. Native model записана как
`u(x,t,d)`, где `d=t−r`, поэтому JVP tangent — **`(v,1,1)`**, а не `(v,1,0)`.
Identity target: `stopgrad(v−d*JVP[u;(v,1,1)])`. 75% samples имеют d=0 и FM target;
у 25% две независимые sigmoid(N(−0.4,1)) сортируются в r≤t. JVP строится из detached
parameters: через target нет higher-order optimization; live prediction имеет обычный
parameter gradient. Adaptive L2 weights `1/(SUM(error²).detach()+0.001)^1` также detached. Здесь именно SUM по state coordinates, как в official MeanFlow code;
замена SUM на MEAN меняет effective c для D=2 и D=60.

In [ ]:
def train_meanflow(ctx):
    for _ in update_range(ctx):
        clean, noise, t, conditions = data_batch(ctx)
        batch = len(clean)
        fm_count = int(batch * ctx.config["meanflow_fm_fraction"])
        sampled = torch.randn(2, batch, generator=ctx.rng["duration"]).to(clean.device)
        sampled = torch.sigmoid(
            ctx.config["meanflow_logit_mean"] + ctx.config["meanflow_logit_std"] * sampled
        )
        r, t = sampled.sort(dim=0).values
        duration = t - r
        duration[:fm_count] = 0
        anchor = (1 - expand_time(t, clean)) * clean + expand_time(t, clean) * noise
        conditional_velocity = (noise - clean).detach()
        target = conditional_velocity.clone()
        x = anchor[fm_count:]
        tt = t[fm_count:]
        dd = duration[fm_count:]
        cond = conditions[fm_count:]
        # A detached functional model keeps the JVP target out of parameter optimization.
        parameters = {name: value.detach() for name, value in ctx.model.named_parameters()}
        buffers = {name: value.detach() for name, value in ctx.model.named_buffers()}

        def average_velocity(xx, when, width):
            return torch.func.functional_call(
                ctx.model,
                (parameters, buffers),
                (xx, when),
                {"conditions": cond, "step_size": width},
            ).sample

        _, derivative = torch.func.jvp(
            average_velocity,
            (x, tt, dd),
            (conditional_velocity[fm_count:], torch.ones_like(tt), torch.ones_like(dd)),
        )
        target[fm_count:] = (
            conditional_velocity[fm_count:] - expand_time(dd, x) * derivative
        ).detach()
        prediction = call_velocity(ctx.model, anchor, t, conditions, width=duration)
        squared_sum = (prediction - target.detach()).flatten(1).square().sum(dim=1)
        weight = (squared_sum.detach() + ctx.config["meanflow_adaptive_c"]).pow(
            -ctx.config["meanflow_adaptive_p"]
        )
        loss = (weight * squared_sum).mean()
        take_step(ctx, loss, {"fm_examples": fm_count, "meanflow_examples": batch - fm_count})

In [ ]:
def inference_exports(ctx):
    exports = {}
    for nfe in (1, 2, 4):
        if ctx.method == "progressive":
            folder = f"stage{nfe}/ema_model"
            manifest = json.loads((ctx.directory / f"stage{nfe}/manifest.json").read_text())
            trained = nfe
            kind = "native"
            cumulative = manifest["cumulative_updates_at_export"]
            sha = manifest["ema_sha256"]
        else:
            folder = "ema_model"
            sha = model_hash(ctx.ema)
            cumulative = ctx.completed
            trained = 1 if ctx.method in ("endpoint", "consistency") else None
            kind = "off_target" if ctx.method == "endpoint" and nfe > 1 else "native"
        exports[str(nfe)] = {
            "path": folder,
            "sha256": sha,
            "trained_nfe": trained,
            "native_or_off_target": kind,
            "cumulative_updates_at_export": cumulative,
        }
    return exports


def train_method(
    method,
    data,
    teacher,
    seed,
    directory,
    config,
    bank,
    bank_metadata,
    *,
    source_sha256,
    resume=False,
    stop_after=None,
):
    if method not in METHODS:
        raise ValueError("Unknown method")
    ctx = start_context(
        method,
        data,
        teacher,
        seed,
        config,
        directory,
        source_sha256,
        bank,
        bank_metadata,
        resume,
        stop_after,
    )
    parent_before = model_hash(teacher)
    checkpoint_context(ctx, "running")
    try:
        if method in ("cfm", "ot_cfm"):
            train_cfm(ctx, optimal_transport=method == "ot_cfm")
        elif method == "endpoint":
            train_endpoint(ctx)
        elif method == "progressive":
            train_progressive(ctx)
        elif method == "reflow":
            train_reflow(ctx)
        elif method == "consistency":
            train_consistency(ctx)
        elif method == "shortcut":
            train_shortcut(ctx)
        elif method == "meanflow":
            train_meanflow(ctx)
        completed = ctx.completed == config["updates"]
        status = "completed" if completed else "interrupted"
        if completed:
            export_model(ctx, ctx.directory)
        checkpoint_context(ctx, status)
        manifest = {
            "schema_version": 1,
            "status": status,
            "task": data["task"],
            "method": method,
            "seed": seed,
            "updates_planned": config["updates"],
            "updates_completed": ctx.completed,
            "model_kind": "interval" if method in ("shortcut", "meanflow") else "velocity",
            "model_class": ctx.ema.__class__.__name__,
            "ema_sha256": model_hash(ctx.ema),
            "init_sha256": ctx.init_sha256,
            "teacher_sha256": parent_before,
            "common_ancestor_sha256": parent_before
            if config["initialization"] == "parent"
            else None,
            "source_sha256": source_sha256,
            "config_sha256": json_digest(config),
            "train_data_sha256": data["train_data_sha256"],
            "train_state_sha256": data["train_state_sha256"],
            "train_conditions_sha256": data["train_conditions_sha256"],
            "condition_spec": data["condition_spec"],
            "condition_dropout": 0.0,
            "cfg": False,
            "whole_support_single_condition": True,
            "route_id_input": False,
            "endpoint_clamped_contract": data["task"] == "wall",
            "state_dimension": math.prod(data["sample_shape"]),
            "pairbank_sha256": bank_metadata["sha256"],
            "pairbank_contract": bank_metadata["contract"],
            "initialization": config["initialization"],
            "warm_start": "same fixed20000-update task parent; neutral interval branch where applicable"
            if config["initialization"] == "parent"
            else "fresh data-model initialization; explicit separate exploratory run",
            "training_seconds": ctx.elapsed + time.perf_counter() - ctx.started,
            "training_cost": ctx.costs,
            "cost_definition": "actual forward calls/sample-evals; not FLOPs. Walltime includes optimizer/backward/JVP. Offline shared pairbank cost is separate.",
            "raw_batch_trace_sha256": json_digest(ctx.raw_trace),
            "pair_indices_trace_sha256": json_digest(ctx.pair_trace),
            "inference_exports": inference_exports(ctx) if completed else {},
            "validation_selection": "final EMA only; no best-checkpoint selection",
        }
        atomic_json(ctx.directory / "manifest.json", manifest)
        atomic_json(ctx.directory / "history.json", ctx.history)
        if model_hash(teacher) != parent_before or any(
            p.grad is not None or p.requires_grad for p in teacher.parameters()
        ):
            raise AssertionError("Frozen common parent changed or received gradients")
        return manifest
    except BaseException as error:
        status = "interrupted" if isinstance(error, KeyboardInterrupt) else "failed"
        checkpoint_context(ctx, status)
        atomic_json(
            ctx.directory / "manifest.json",
            {
                "status": status,
                "method": method,
                "task": data["task"],
                "seed": seed,
                "updates_completed": ctx.completed,
                "updates_planned": config["updates"],
                "error": str(error),
                "teacher_sha256": parent_before,
                "source_sha256": source_sha256,
            },
        )
        raise
    finally:
        for hook in ctx.hooks:
            hook.remove()


def run_zoo(
    config,
    *,
    repo_root=None,
    tasks=None,
    methods=None,
    seeds=None,
    resume=False,
    stop_after=None,
    prepare_only=False,
    dry_run=False,
):
    config = resolved_config(config)
    selected_tasks = config["tasks"] if tasks is None else list(tasks)
    selected_methods = config["methods"] if methods is None else list(methods)
    selected_seeds = config["seeds"] if seeds is None else list(seeds)
    if (
        not set(selected_tasks) <= set(config["tasks"])
        or not set(selected_methods) <= set(config["methods"])
        or not set(selected_seeds) <= set(config["seeds"])
    ):
        raise ValueError("Run selectors must be a subset of the resolved config")
    if dry_run:
        return {
            "status": "dry_run",
            "config": config,
            "config_sha256": json_digest(config),
            "jobs": [
                {"task": task, "method": method, "seed": seed, "updates": config["updates"]}
                for task in selected_tasks
                for seed in selected_seeds
                for method in selected_methods
            ],
            "data_loaded": False,
        }
    if repo_root is None:
        repo_root = next(
            (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/diffusion_lab").is_dir()),
            None,
        )
    if repo_root is None:
        raise ValueError("Pass repo_root for the checkout containing native parents")
    repo_root = Path(repo_root)
    output = Path(config["output_dir"]).expanduser().resolve()
    output.mkdir(parents=True, exist_ok=True)
    # atomic_json uses a deterministic temporary name, so serialize root setup.
    # This lock is brief; independent method/seed training remains parallel.
    import fcntl

    with (output / ".run_config.lock").open("a") as config_lock:
        fcntl.flock(config_lock, fcntl.LOCK_EX)
        config_path = output / "run_config.json"
        if config_path.exists() and json.loads(config_path.read_text()) != config:
            raise ValueError("Output root has a different frozen config")
        if not config_path.exists():
            atomic_json(config_path, config)
    torch.set_num_threads(config["threads"])
    device = choose_device(config["device"])
    source_sha256 = notebook_source_hash(repo_root)
    reports = []
    for task in selected_tasks:
        data = load_task(task, "train")
        teacher, _ = load_parent(task, data, repo_root, device)
        bank, metadata = prepare_pairbank(data, teacher, config, output / task / "pairbank")
        if prepare_only:
            reports.append({"task": task, "bank_sha256": metadata["sha256"], "status": "prepared"})
            continue
        for seed in selected_seeds:
            for method in selected_methods:
                directory = output / task / f"seed{seed}" / method
                report = train_method(
                    method,
                    data,
                    teacher,
                    seed,
                    directory,
                    config,
                    bank,
                    metadata,
                    source_sha256=source_sha256,
                    resume=resume,
                    stop_after=stop_after,
                )
                reports.append(
                    {
                        "task": task,
                        "method": method,
                        "seed": seed,
                        "status": report["status"],
                        "updates_completed": report["updates_completed"],
                        "ema_sha256": report["ema_sha256"],
                        "directory": str(directory),
                    }
                )
    return {
        "status": "completed"
        if all(row["status"] in ("completed", "prepared") for row in reports)
        else "interrupted",
        "config_sha256": json_digest(config),
        "source_sha256": source_sha256,
        "runs": reports,
        "final_data_accessed": False,
    }

## Быстрый запуск и варианты

`CONFIG` — обычный dict. Чтобы менять задачу, conditioner, model, loss или sampling,
редактируйте функции выше; новый source hash отделит эксперимент от прежних checkpoints.
Для main cohort runner читает отдельный config (2000 updates, оба tasks, seeds 161/262/363).
Здесь default 200 updates, только rays и seed 161. Все восемь objective loops остаются видны.
`RUN_TRAINING=False` позволяет сначала прочитать formulas и поменять параметры.

Native parent и dataset исходники скачиваются из того же checkout. Установка не встраивает
wheel/weights: при standalone kernel клонируется source repo, native files читаются отдельно.

In [ ]:
import tempfile

CONFIG = copy.deepcopy(DEFAULT_CONFIG)
RUN_TRAINING = False
RESUME = False
REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/diffusion_lab").is_dir()), None
)
if REPO_ROOT is None:
    REPO_ROOT = Path(tempfile.mkdtemp(prefix="method-zoo-source-")) / "repository"
    subprocess.check_call(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            GIT_BRANCH,
            "https://github.com/Vovanm88/ddpm_vibe_research.git",
            str(REPO_ROOT),
        ]
    )
CONFIG = resolved_config(CONFIG)
print(json.dumps(run_zoo(CONFIG, repo_root=REPO_ROOT, dry_run=True), indent=2))
if RUN_TRAINING:
    RUN_REPORT = run_zoo(CONFIG, repo_root=REPO_ROOT, resume=RESUME)
    print(json.dumps(RUN_REPORT, indent=2))

## Validation и cached interactive inspection

Эта ячейка использует **validation**, не новый final bank. Для rays один и тот же latent
повторяется для 16 target IDs; для wall — для всех held-out scenes. CM extra-noise bank тоже
повторяется между conditions и seeds и используется prefix для 1/2/4 calls. Все выбранные
native EMA exports уже сохранены; widget не вызывает модель при движении slider.

CFM/SWD fidelity к common parent не доказывает геометрическую правильность. Для rays смотрите
requested-target routing и outer coverage; для wall — exact segment collisions и обе моды.
Raw samples не фильтруются перед metrics. Stage-matched progressive export может иметь меньше
cumulative updates — это выводится явно.

In [ ]:
if RUN_TRAINING:
    import ipywidgets as widgets
    import matplotlib.pyplot as plt
    from IPython.display import clear_output, display

    from diffusion_lab.conditional_metrics import conditional_routing_metrics
    from diffusion_lab.environments.multimodal_navigation import evaluate_navigation
    from diffusion_lab.navigation_visualization import plot_navigation_paths

    TASK = CONFIG["tasks"][0]
    SEED = CONFIG["seeds"][0]
    EVAL_NFES = (1, 2, 4)
    MAX_NFE = max(EVAL_NFES)
    PREVIEW_PER_CONDITION = 32
    validation = load_task(TASK, "validation")
    condition_values = torch.cat(
        [validation["conditions"][pool[:1]] for pool in validation["pools"]]
    )
    noise_seed = 830001 if TASK == "rays" else 613001
    extra_seed = 830002 if TASK == "rays" else 613002
    base = torch.randn(
        PREVIEW_PER_CONDITION,
        *validation["sample_shape"],
        generator=torch.Generator().manual_seed(noise_seed),
    )
    conditions = condition_values.repeat_interleave(PREVIEW_PER_CONDITION, dim=0)
    initial = base.repeat(len(condition_values), *([1] * len(validation["sample_shape"])))
    extra_base = torch.randn(
        MAX_NFE - 1,
        PREVIEW_PER_CONDITION,
        *validation["sample_shape"],
        generator=torch.Generator().manual_seed(extra_seed),
    )
    extra = extra_base.repeat(1, len(condition_values), *([1] * len(validation["sample_shape"])))
    generations = {}
    for method in CONFIG["methods"]:
        directory = Path(CONFIG["output_dir"]) / TASK / f"seed{SEED}" / method
        manifest = json.loads((directory / "manifest.json").read_text())
        for nfe in EVAL_NFES:
            export = manifest["inference_exports"][str(nfe)]
            model = load_zoo_model(directory / export["path"], device=CONFIG["device"])
            result = sample_zoo(
                model,
                initial,
                conditions,
                nfe=nfe,
                extra_noise=extra[: nfe - 1],
                record_history=True,
            )
            generations[(method, nfe)] = result
            metrics = (
                conditional_routing_metrics(result["samples"], conditions, initial, n_classes=16)
                if TASK == "rays"
                else evaluate_navigation(result["samples"], conditions)
            )
            print(
                method,
                "actual NFE",
                result["nfe"],
                "cumulative updates",
                export["cumulative_updates_at_export"],
                "routing" if TASK == "rays" else "success",
                metrics["accuracy_all"] if TASK == "rays" else metrics["success_fraction"],
            )
    method_control = widgets.Dropdown(options=CONFIG["methods"], description="Method")
    nfe_control = widgets.Dropdown(options=EVAL_NFES, value=4, description="NFE")
    condition_control = widgets.Dropdown(
        options=list(range(len(condition_values))),
        description="Target" if TASK == "rays" else "Scene",
    )
    noise_control = widgets.IntSlider(
        value=0,
        min=0,
        max=PREVIEW_PER_CONDITION - 1,
        step=1,
        description="Noise",
        continuous_update=False,
    )
    frame_control = widgets.IntSlider(
        value=0, min=0, max=2 * MAX_NFE - 1, step=1, description="Event", continuous_update=False
    )
    play = widgets.Play(value=0, min=0, max=2 * MAX_NFE - 1, step=1, interval=180)
    link = widgets.jslink((play, "value"), (frame_control, "value"))
    canvas = widgets.Output()

    def show_cached(change=None):
        result = generations[(method_control.value, nfe_control.value)]
        history = result["history"]
        last = len(history) - 1
        frame_control.max = play.max = last
        frame = min(frame_control.value, last)
        condition_id = condition_control.value
        index = condition_id * PREVIEW_PER_CONDITION + noise_control.value
        with canvas:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(7, 6))
            if TASK == "rays":
                real = validation["raw_samples"][validation["pools"][condition_id]]
                ax.scatter(
                    real[:, 0],
                    real[:, 1],
                    s=4,
                    alpha=0.2,
                    color="grey",
                    label="requested target reference",
                )
                cached = history[:, index]
                colors = plt.get_cmap("rainbow_r")(torch.linspace(0, 1, len(cached)).numpy())
                ax.plot(cached[:, 0], cached[:, 1], color="grey", alpha=0.4)
                ax.scatter(cached[:, 0], cached[:, 1], c=colors, s=25)
                ax.scatter(*cached[frame], color="black", s=70)
                ax.set(xlim=(-3, 3), ylim=(-3, 3), aspect="equal")
            else:
                cond = condition_values[condition_id]
                plot_navigation_paths(
                    history[frame, index : index + 1],
                    walls=cond[4:].reshape(1, 2, 2),
                    start=cond[:2],
                    goal=cond[2:4],
                    color_by="progress",
                    ax=ax,
                )
            kind = result["event_kinds"][frame]
            ax.set_title(
                f"{method_control.value}; condition {condition_id}; event {frame}/{last}: {kind}, t={float(result['times'][frame]):.3f}"
            )
            plt.show()

    for control in (method_control, nfe_control, condition_control, noise_control, frame_control):
        control.observe(show_cached, names="value")
    display(
        widgets.VBox(
            [
                widgets.HBox([method_control, nfe_control, condition_control, noise_control]),
                widgets.HBox([play, frame_control]),
                canvas,
            ]
        )
    )
    show_cached()

In [ ]:
# Optional export of ACTUAL cached sampling events, including CM re-noising.
# The frame clock is sampler time, not physical particle travel along a wall path.
EXPORT_GIF = False
if RUN_TRAINING and EXPORT_GIF:
    from matplotlib.animation import FuncAnimation, PillowWriter

    selected_method = method_control.value
    selected_nfe = nfe_control.value
    selected_condition = condition_control.value
    selected_noise = noise_control.value
    result = generations[(selected_method, selected_nfe)]
    index = selected_condition * PREVIEW_PER_CONDITION + selected_noise
    cached = result["history"][:, index]
    figure, axis = plt.subplots(figsize=(7, 6))
    extent = max(1.1, float(cached.abs().max()) + 0.15)

    def draw_cached_event(frame):
        axis.clear()
        if TASK == "rays":
            real = validation["raw_samples"][validation["pools"][selected_condition]]
            axis.scatter(real[:, 0], real[:, 1], s=4, alpha=0.2, color="grey")
            colors = plt.get_cmap("rainbow_r")(torch.linspace(0, 1, len(cached)).numpy())
            axis.plot(cached[: frame + 1, 0], cached[: frame + 1, 1], color="grey", alpha=0.5)
            axis.scatter(
                cached[: frame + 1, 0], cached[: frame + 1, 1], c=colors[: frame + 1], s=40
            )
            axis.scatter(*cached[frame], color="black", s=70)
            axis.set(xlim=(-extent, extent), ylim=(-extent, extent), aspect="equal")
        else:
            cond = condition_values[selected_condition]
            plot_navigation_paths(
                cached[frame : frame + 1],
                walls=cond[4:].reshape(1, 2, 2),
                start=cond[:2],
                goal=cond[2:4],
                color_by="progress",
                ax=axis,
                bounds=((-extent, extent), (-extent, extent)),
            )
        axis.set_title(
            f"{selected_method} NFE {selected_nfe}: actual event {frame}/{len(cached) - 1}\n"
            f"{result['event_kinds'][frame]}, t={float(result['times'][frame]):.3f}; sampler events"
        )

    animation = FuncAnimation(figure, draw_cached_event, frames=len(cached), interval=180)
    gif_path = (
        Path(CONFIG["output_dir"]) / f"{TASK}_{selected_method}_N{selected_nfe}_actual_events.gif"
    )
    animation.save(gif_path, writer=PillowWriter(fps=5))
    plt.close(figure)
    print(gif_path)

## Primary references и исследовательские границы

- [Flow Matching](https://arxiv.org/abs/2210.02747), [OT-CFM](https://arxiv.org/abs/2302.00482)
- [Progressive Distillation](https://arxiv.org/abs/2202.00512)
- [Rectified Flow / reflow](https://arxiv.org/abs/2209.03003)
- [Consistency Models](https://arxiv.org/abs/2303.01469)
- [Shortcut Models](https://arxiv.org/abs/2410.12557)
- [MeanFlow](https://arxiv.org/abs/2505.13447)

Все primary recipes адаптированы к текущему decreasing-time convention и conditional toy
parent. Native interval model сохраняет прежний conditioner; learned duration branch добавляет
параметры и отражается в manifests. Публикуются selected EMA, а optimizer/RNG histories
остаются отдельными private checkpoints. Comparison не делает выводов о full-scale image models.
Для benchmark ranking используйте отдельный frozen-cohort evaluator, а не визуальный выбор
удачного seed, траектории, Euler budget или лучшего промежуточного checkpoint.